# 01 - Chinese Speech Transcription (ASR)

This notebook uses **faster-whisper** with the **large-v3** model to transcribe Chinese audio from the source videos.

**Requirements:** Google Colab with GPU runtime (T4 is sufficient).

**Output:** JSON transcript files with word-level timestamps saved to Google Drive.

In [ ]:
# Install dependencies
!pip install -q faster-whisper

In [ ]:
# Mount Google Drive for input/output
from google.colab import drive
drive.mount('/content/drive')

# Set paths - adjust to your Drive folder structure
import os
PROJECT_DIR = '/content/drive/MyDrive/AIVideoLanguageTransformation'
AUDIO_DIR = os.path.join(PROJECT_DIR, 'data', 'audio', 'original')
TRANSCRIPTS_DIR = os.path.join(PROJECT_DIR, 'data', 'transcripts')
os.makedirs(TRANSCRIPTS_DIR, exist_ok=True)

print(f'Audio dir: {AUDIO_DIR}')
print(f'Transcripts dir: {TRANSCRIPTS_DIR}')
print(f'Audio files: {os.listdir(AUDIO_DIR) if os.path.exists(AUDIO_DIR) else "NOT FOUND"}')

In [ ]:
from faster_whisper import WhisperModel
import json
import time

# Load model (downloads on first run, ~3GB)
print('Loading Whisper large-v3...')
model = WhisperModel('large-v3', device='cuda', compute_type='float16')
print('Model loaded.')

In [ ]:
def transcribe_audio(audio_path, language='zh', beam_size=5):
    """Transcribe an audio file and return segments with word-level timestamps."""
    print(f'\nTranscribing: {os.path.basename(audio_path)}')
    start = time.time()

    segments, info = model.transcribe(
        audio_path,
        language=language,
        beam_size=beam_size,
        word_timestamps=True,
        vad_filter=True,           # Filter out non-speech
        vad_parameters=dict(
            min_silence_duration_ms=500,
        ),
    )

    print(f'Detected language: {info.language} (prob: {info.language_probability:.2f})')
    print(f'Audio duration: {info.duration:.1f}s')

    result = []
    for i, seg in enumerate(segments):
        words = []
        if seg.words:
            words = [
                {'word': w.word.strip(), 'start': round(w.start, 3), 'end': round(w.end, 3), 'probability': round(w.probability, 3)}
                for w in seg.words
            ]

        segment_data = {
            'id': i,
            'text_zh': seg.text.strip(),
            'start': round(seg.start, 3),
            'end': round(seg.end, 3),
            'words': words,
        }
        result.append(segment_data)
        print(f'  [{seg.start:.1f}s - {seg.end:.1f}s] {seg.text.strip()}')

    elapsed = time.time() - start
    print(f'\nTranscription complete: {len(result)} segments in {elapsed:.1f}s')
    return result

In [ ]:
# Transcribe all audio files
audio_files = sorted([f for f in os.listdir(AUDIO_DIR) if f.endswith('.wav')])
print(f'Found {len(audio_files)} audio files to transcribe')

for audio_file in audio_files:
    audio_path = os.path.join(AUDIO_DIR, audio_file)
    transcript = transcribe_audio(audio_path)

    # Save transcript
    stem = os.path.splitext(audio_file)[0]
    output_path = os.path.join(TRANSCRIPTS_DIR, f'{stem}_zh.json')
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(transcript, f, ensure_ascii=False, indent=2)
    print(f'Saved: {output_path}')

print('\nAll transcriptions complete!')

In [ ]:
# Quick review: print all transcripts
for json_file in sorted(os.listdir(TRANSCRIPTS_DIR)):
    if not json_file.endswith('.json'):
        continue
    path = os.path.join(TRANSCRIPTS_DIR, json_file)
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f'\n=== {json_file} ({len(data)} segments) ===')
    for seg in data[:5]:  # Show first 5 segments
        print(f"  [{seg['start']:.1f}s-{seg['end']:.1f}s] {seg['text_zh']}")
    if len(data) > 5:
        print(f'  ... and {len(data) - 5} more segments')